# Phase 4: Causal Patching — Activation Patching

**Experimental protocol (Denoising):**
1. Run **clean prompt** → save clean hidden states for all layers (at the last token of `<hop1>`)
2. Run **counterfactual (corrupted) prompt** normally → save the baseline (hallucinated) answer
3. For **each layer**: Run **counterfactual prompt again**, but REPLACE the layer's hidden state (at the last token of `<hop1>`) with the CLEAN state harvested in step 1
4. Measure whether injecting the clean state successfully restores the correct answer

If the correct answer is restored in ≥50% of cases for a given layer, that layer is causally responsible for the hop-1 reasoning step.

In [ ]:
!pip install -q transformers accelerate huggingface_hub
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("Logged in via Kaggle Secrets!")
except Exception as e:
    print(f"Kaggle Secrets failed ({e}), using fallback...")
    hf_token = "REPLACE_ME_WITH_YOUR_RAW_HF_TOKEN"
    huggingface_hub.login(token=hf_token)

In [ ]:
import json, torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print(f"Model loaded. Device: {model.device}")

In [ ]:
print("Downloading dataset...")
file_path = hf_hub_download(
    repo_id="AnishRacherla/LinguaFranca-Phase3",
    filename="data/2wikimultihopqa/augmented.jsonl",
    repo_type="dataset"
)
with open(file_path) as f:
    data = [json.loads(line) for line in f]

counterfactuals = [d for d in data if d.get("is_counterfactual", False)]
cleans = {d["id"]: d for d in data if not d.get("is_counterfactual", False)}
print(f"Loaded {len(cleans)} clean examples, {len(counterfactuals)} counterfactuals.")

# Build minimal pairs: clean example answered correctly (hop1 label=0) 
# AND its CF twin answered incorrectly (hop1 label=1)
pairs = []
for cf in counterfactuals:
    clean = cleans.get(cf.get("clean_pair_id"))
    if not clean:
        continue
    clean_h1 = next((h for h in clean.get("hops", []) if h["hop_idx"] == 1), None)
    cf_h1 = next((h for h in cf.get("hops", []) if h["hop_idx"] == 1), None)
    if clean_h1 and cf_h1 and clean_h1.get("label") == 0 and cf_h1.get("label") == 1:
        pairs.append((clean, cf))
            

print(f"Found {len(pairs)} perfect minimal pairs.")

In [ ]:
def fuzzy_match(expected, actual):
    import re
    expected_words = set(re.findall(r'\w+', expected.lower()))
    actual_words = set(re.findall(r'\w+', actual.lower()))
    return expected_words.issubset(actual_words) if expected_words else False

def extract_answer_tag(text):
    """Pull out just the <answer>...</answer> part."""
    import re
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if m:
        return m.group(1).strip().lower()
    # Fallback: last non-empty line
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    return lines[-1].lower() if lines else ""

num_layers = model.config.num_hidden_layers
print(f"Running causal patching (Entity-Level) on {len(pairs)} pairs across all {num_layers} layers...")

# We will do DENOISING: Patching CLEAN state into CORRUPTED prompt to restore correctness.
layer_success_counts = {l: 0 for l in range(num_layers)}
n_total = 0
examples_log = []

for i, (clean, cf) in enumerate(tqdm(pairs, desc="Causal Patching")):
    # 1. Get the text up to the end of Hop 1
    import re
    def get_hop1(cot):
        m = re.search(r'(<hop1>.*?</hop1>)', cot, re.DOTALL)
        return m.group(1) if m else ""
    
    clean_h1_text = get_hop1(clean["generated_cot"])
    cf_h1_text = get_hop1(cf["generated_cot"])
    
    if not clean_h1_text or not cf_h1_text:
        continue
        
    clean_full_prompt = clean["prompt"] + clean_h1_text
    cf_full_prompt = cf["prompt"] + cf_h1_text
    
    # 2. HARVEST the CLEAN state at the end of Hop 1 for ALL layers
    clean_states = {}
    def get_harvest_hook(layer_idx):
        def hook(module, args, output):
            hidden = output[0] if isinstance(output, tuple) else output
            # Grab the very last token of the prefill (which is </hop1>)
            clean_states[layer_idx] = hidden[0, -1, :].clone().detach()
            return output
        return hook
    
    clean_inputs = tokenizer(clean_full_prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    handles = []
    for l in range(num_layers):
        handles.append(model.model.layers[l].register_forward_hook(get_harvest_hook(l)))
        
    with torch.no_grad():
        model(**clean_inputs)
        
    for h in handles:
        h.remove()
    
    # 3. Get BASELINE answer for CF (Should be WRONG, as it's a hallucination)
    cf_inputs = tokenizer(cf_full_prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    
    terminators = [tokenizer.eos_token_id]
    if "<|eot_id|>" in tokenizer.vocab:
        terminators.append(tokenizer.convert_tokens_to_ids("<|eot_id|>"))
    elif "<|im_end|>" in tokenizer.vocab:
        terminators.append(tokenizer.convert_tokens_to_ids("<|im_end|>"))
        
    with torch.no_grad():
        baseline_out = model.generate(
            **cf_inputs, 
            min_new_tokens=5, 
            max_new_tokens=250, 
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id
        )
    baseline_text = tokenizer.decode(baseline_out[0][cf_inputs.input_ids.shape[1]:], skip_special_tokens=True)
    baseline_ans = extract_answer_tag(baseline_text)
    
    expected_clean_ans = clean.get("predicted_answer", clean.get("answer", "")).lower()
    
    pair_log = {
        "clean_q": clean["question"],
        "cf_q": cf["question"],
        "baseline_ans": baseline_ans,
        "layer_results": {}
    }
    
    # 4. INJECT the CLEAN state into the CF generation for EACH layer
    for l in range(num_layers):
        clean_vec = clean_states[l]
        injected = [False]
        def inject_hook(module, args, output):
            hidden = output[0] if isinstance(output, tuple) else output
            rest = output[1:] if isinstance(output, tuple) else None
            
            # We only inject ONCE during the prefill of the CF prompt
            if not injected[0] and hidden.shape[1] > 1:
                hidden = hidden.clone()
                hidden[0, -1, :] = clean_vec  # Inject clean vector at </hop1>
                injected[0] = True
                
            if rest is not None:
                return (hidden,) + rest
            return hidden
        
        inject_handle = model.model.layers[l].register_forward_hook(inject_hook)
        with torch.no_grad():
            corrupted_out = model.generate(
                **cf_inputs, 
                min_new_tokens=5, 
                max_new_tokens=250, 
                eos_token_id=terminators,
                pad_token_id=tokenizer.eos_token_id
            )
        inject_handle.remove()
        
        corrupted_text = tokenizer.decode(corrupted_out[0][cf_inputs.input_ids.shape[1]:], skip_special_tokens=True)
        corrupted_ans = extract_answer_tag(corrupted_text)
        
        # Check if the injection RESTORED the correct answer!
        restored = False
        if fuzzy_match(expected_clean_ans, corrupted_ans) and not fuzzy_match(expected_clean_ans, baseline_ans):
            restored = True
            layer_success_counts[l] += 1
            
        pair_log["layer_results"][l] = {
            "corrupted_ans": corrupted_ans,
            "restored": restored
        }
        
    n_total += 1
    examples_log.append(pair_log)
    
    if i < 3:
        # Sanity check: generate from clean_full_prompt to see if it also yields blank
        with torch.no_grad():
            clean_check_out = model.generate(
                **clean_inputs, 
                min_new_tokens=5,
                max_new_tokens=250, 
                eos_token_id=terminators,
                pad_token_id=tokenizer.eos_token_id
            )
        clean_check_text = tokenizer.decode(clean_check_out[0][clean_inputs.input_ids.shape[1]:], skip_special_tokens=True)
        clean_check_ans = extract_answer_tag(clean_check_text)

        print(f"\n--- Example {i+1} ---")
        print(f"CF Q:             {cf['question']}")
        print(f"Clean Expected:   {expected_clean_ans}")
        print(f"Clean Gen Ans:    {clean_check_ans}")
        print(f"CF Baseline Ans:  {baseline_ans}")
        restored_layers = [l for l in range(num_layers) if pair_log["layer_results"][l]["restored"]]
        print(f"Restored on layers: {restored_layers if restored_layers else 'None'}")

print(f"\n{'='*40}")
print(f"=== CAUSAL PATCHING (DENOISING) RESULTS ===")
print(f"{'='*40}")
print(f"Total pairs tested:       {n_total}\n")
print("Denoising Success Rate by Layer:")
rates = []
for l in range(num_layers):
    rate = layer_success_counts[l] / n_total * 100 if n_total > 0 else 0
    rates.append(rate)
    print(f"Layer {l:02d}: {rate:5.1f}% ({layer_success_counts[l]}/{n_total})", end=" ")
    if rate >= 50:
        print("✅ STRONG")
    elif rate >= 20:
        print("⚠️ MODERATE")
    else:
        print("❌ WEAK")
        
# Plot the results
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(range(num_layers), rates, marker='o', linewidth=2)
plt.axhline(y=50, color='r', linestyle='--', alpha=0.7, label='Strong Effect Threshold (50%)')
plt.axhline(y=20, color='orange', linestyle='--', alpha=0.7, label='Moderate Effect Threshold (20%)')
plt.xlabel("Layer")
plt.ylabel("Denoising Success Rate (%)")
plt.title("Causal Patching: Restoration of Correct Answer by Layer")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()